# `tf.llm` — Open-Source LLM Loading & LoRA Fine-Tuning in TensorFlow

This notebook demonstrates the new `tf.llm` submodule that makes it trivial
to load any open-source LLM and fine-tune it with LoRA — all within the
familiar TensorFlow / Keras ecosystem.

**Supported models (initial release):**
- `meta-llama/Llama-3`, `Llama-3.1`, `Llama-3.2`, `Llama-2`
- `google/gemma-2b`, `gemma-7b`, `gemma-2`
- `mistralai/Mistral-7B`, `Mixtral-8x7B`
- `tiiuae/falcon-7b`, `falcon-40b`
- Any HuggingFace `AutoModelForCausalLM` (generic fallback)


## 1. Installation

In [ ]:
# tf.llm requires transformers, peft, and accelerate
# (torch is already present in most GPU environments)
!pip install transformers peft accelerate -q

## 2. Load a Pretrained LLM — One Line

In [ ]:
import tensorflow as tf

# Load Gemma 2B (no token required — fully open-access)
model = tf.llm.from_pretrained(
    "google/gemma-2b",
    dtype="bfloat16",    # Use bfloat16 to halve VRAM usage
    device_map="auto",  # Spread across all available GPUs automatically
)

print(model)

In [ ]:
# For gated models like Llama-3, pass your HuggingFace token
# (or set the HUGGINGFACE_TOKEN environment variable)
#
# model = tf.llm.from_pretrained(
#     "meta-llama/Llama-3",
#     token="hf_...",
#     dtype="bfloat16",
# )

## 3. Inference Before Fine-Tuning

In [ ]:
# Generate text from a prompt
response = model.generate(
    "Explain quantum entanglement in simple terms:",
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
)
print(response)

In [ ]:
# Batch generation — pass a list of prompts
prompts = [
    "The capital of Bangladesh is",
    "TensorFlow was created by",
    "The best way to learn machine learning is",
]
responses = model.generate(prompts, max_new_tokens=50, do_sample=False)
for p, r in zip(prompts, responses):
    print(f"Prompt : {p}")
    print(f"Output : {r}")
    print()

## 4. LoRA Fine-Tuning — 3 Lines

In [ ]:
from datasets import load_dataset

# Load a small instruction-tuning dataset
raw = load_dataset("tatsu-lab/alpaca", split="train[:1000]")

# Tokenize into a format the model understands
def tokenize(example):
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    tokens = model._tokenizer(
        text, truncation=True, max_length=512, padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = raw.map(tokenize, remove_columns=raw.column_names)
dataset.set_format("torch")
print(f"Dataset size: {len(dataset)} examples")

In [ ]:
# --- The 3-line fine-tune ---

model.compile(optimizer="adam", loss="causal_lm")  # Applies LoRA automatically
history = model.fit(dataset, epochs=1)              # Fine-tune

print(f"Final loss: {history['loss'][-1]:.4f}")

In [ ]:
# Inspect what LoRA did to the model
model.summary()

## 5. Advanced: Custom LoRA Configuration

In [ ]:
# Load a fresh copy of the model for advanced fine-tuning
model_adv = tf.llm.from_pretrained("google/gemma-2b", dtype="bfloat16")

model_adv.compile(
    optimizer="adamw",
    loss="causal_lm",
    lora_config={
        "r": 32,                                          # Higher rank → more capacity
        "lora_alpha": 64,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention
        "lora_dropout": 0.1,
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
)

model_adv.summary()

## 6. Save and Reload

In [ ]:
# Save the fine-tuned model (weights + tokenizer + tf_llm_meta.json)
model.save_pretrained("./my_gemma_finetuned")
print("Model saved!")

In [ ]:
# Reload from disk — works exactly like from_pretrained with an HF model ID
model_reloaded = tf.llm.from_pretrained("./my_gemma_finetuned")
print(model_reloaded)

# Verify generation still works
print(model_reloaded.generate("Hello, world!", max_new_tokens=30))

## 7. Comparison with Raw HuggingFace (before `tf.llm`)

| Task | Raw HuggingFace | `tf.llm` |
|------|----------------|----------|
| Load model | 10–15 lines | **1 line** |
| Apply LoRA | 15–20 lines | **0 lines** (auto in `compile`) |
| Fine-tune | 30–50 lines | **1 line** (`model.fit`) |
| Generate text | 5–8 lines | **1 line** |
| Save model | 3–5 lines | **1 line** |
| **Total** | **~65–100 lines** | **~5 lines** |

The underlying engine is identical — `tf.llm` is a zero-overhead abstraction.

## 8. Supported Models Quick Reference

```python
# LLaMA family
tf.llm.from_pretrained("meta-llama/Llama-3",          token="hf_...")
tf.llm.from_pretrained("meta-llama/Llama-3.1-8B",     token="hf_...")
tf.llm.from_pretrained("meta-llama/Llama-2-7b-hf",    token="hf_...")

# Gemma family (no token needed)
tf.llm.from_pretrained("google/gemma-2b")
tf.llm.from_pretrained("google/gemma-7b")
tf.llm.from_pretrained("google/gemma-2-9b")

# Mistral family
tf.llm.from_pretrained("mistralai/Mistral-7B-v0.1")
tf.llm.from_pretrained("mistralai/Mixtral-8x7B-v0.1")

# Falcon family
tf.llm.from_pretrained("tiiuae/falcon-7b")

# Generic fallback — any HuggingFace causal LM
tf.llm.from_pretrained("microsoft/phi-3-mini-4k-instruct")
tf.llm.from_pretrained("Qwen/Qwen2-7B")
```